# 🏭 Cost-Sensitive Semiconductor Yield Prediction with Dual XAI

**Author:** Ahsan Zaman

---

## Objective

In semiconductor manufacturing, **shipping a defective chip (False Negative)** carries an
asymmetrically higher cost than re-testing a good chip (False Positive). This notebook
builds a **cost-sensitive classification pipeline** on the UCI SECOM dataset that:

| Aspect | Approach |
|---|---|
| **Imbalance** (~93 % Pass / ~7 % Fail) | SMOTE oversampling **+** `scale_pos_weight` in XGBoost |
| **Baseline model** | Random Forest (class-weighted) |
| **Champion model** | XGBoost with Bayesian-tuned cost-sensitive weighting |
| **Primary metric** | **F-Beta (β = 2)** — heavily penalises missed defects |
| **Cost evaluation** | Custom cost-matrix analysis (FN cost ≫ FP cost) |
| **Global XAI** | SHAP (TreeExplainer) — feature importance across the whole test set |
| **Local XAI** | LIME — explaining *why* a single chip was predicted to fail |

### Dataset
The **SECOM dataset** from the UCI Machine Learning Repository contains 1,567 observations
of 590 sensor signals collected during semiconductor fabrication, with a binary Pass/Fail label.

> **Key challenges:** extreme class imbalance, ~41 % of cells contain missing values,
> high-dimensional feature space with many constant/redundant sensors.

---
## 1 · Environment Setup & Data Fetching

We install all required packages and download the SECOM dataset directly from Kaggle
using the Kaggle CLI. No local CSV files are assumed.

In [ ]:
# ── 1.1  Install dependencies ──────────────────────────────────────────────────
!pip install -q kaggle xgboost shap lime imbalanced-learn scikit-learn pandas numpy \
    matplotlib seaborn

In [ ]:
# ── 1.2  Configure Kaggle credentials ─────────────────────────────────────────
# In Google Colab, upload your kaggle.json or set the env vars below.
# Option A – upload via Colab files pane:
#   from google.colab import files
#   files.upload()  # upload kaggle.json
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B – paste your credentials directly (⚠ do not commit to Git):
import os
# os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'
# os.environ['KAGGLE_KEY']      = 'YOUR_KEY'

# Verify Kaggle CLI is accessible
!kaggle --version

In [ ]:
# ── 1.3  Download & extract the SECOM dataset from Kaggle ─────────────────────
!kaggle datasets download -d paresh2047/uci-semcom -p ./data --unzip --force
!ls ./data

In [ ]:
# ── 1.4  Core imports ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer        # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    fbeta_score, precision_score, recall_score, f1_score,
    make_scorer
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
import shap
import lime
import lime.lime_tabular

# Aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold'
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('✅ All imports successful.')

---
## 2 · Data Loading & Initial Exploration

The SECOM CSV bundles a *Time* column, 590 sensor features, and a *Pass/Fail* label.
We convert the label so that **1 = Fail (defect)** and **0 = Pass**.

In [ ]:
# ── 2.1  Load the dataset ─────────────────────────────────────────────────────
import glob

# Auto-detect the CSV file in the data directory
csv_candidates = glob.glob('./data/*.csv')
if csv_candidates:
    csv_path = csv_candidates[0]
    print(f'Loading from CSV: {csv_path}')
    raw = pd.read_csv(csv_path, na_values=['NaN', 'nan', '', 'NA'])

    # Separate timestamp, features, and label
    timestamps = pd.to_datetime(raw['Time'], errors='coerce')
    y_full     = (raw['Pass/Fail'] == 1).astype(int)           # 1 = Fail
    feature_cols = [c for c in raw.columns if c not in ('Time', 'Pass/Fail')]
    X_full = raw[feature_cols].copy()
    X_full.columns = [f'Feature_{i}' for i in range(X_full.shape[1])]
else:
    # Fallback: separate .data files
    X_full = pd.read_csv('./data/secom.data', sep=r'\s+', header=None,
                         na_values=['NaN', 'nan', ''])
    X_full.columns = [f'Feature_{i}' for i in range(X_full.shape[1])]
    labels_df = pd.read_csv('./data/secom_labels.data', sep=r'\s+',
                            header=None, names=['Label', 'Timestamp'])
    y_full     = (labels_df['Label'] == 1).astype(int)
    timestamps = pd.to_datetime(labels_df['Timestamp'], errors='coerce')

print(f'Observations : {X_full.shape[0]:,}')
print(f'Features     : {X_full.shape[1]:,}')
print(f'Pass (0)     : {(y_full == 0).sum():,}  ({(y_full == 0).mean():.1%})')
print(f'Fail (1)     : {(y_full == 1).sum():,}  ({(y_full == 1).mean():.1%})')

### 2.2 · Class Distribution Visualisation

The extreme **~93 % / 7 %** split means a naïve "always predict Pass" classifier
would score >93 % accuracy — but miss *every* defective chip. Accuracy is therefore
a misleading metric; we must use **Recall**, **F-Beta**, and **PR-AUC** instead.

In [ ]:
# ── 2.2  Visualise class imbalance ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

counts = y_full.value_counts().sort_index()
colors = ['#2ecc71', '#e74c3c']

# Bar plot
bars = axes[0].bar(['Pass (0)', 'Fail (1)'], counts.values, color=colors,
                    edgecolor='white', linewidth=1.2)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, cnt + 15,
                 f'{cnt}\n({cnt / len(y_full):.1%})',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')

# Pie chart
axes[1].pie(counts.values, labels=['Pass', 'Fail'], autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion')

plt.suptitle('Severe Class Imbalance in SECOM Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.3 · Missing-Value Landscape

Many sensors have significant gaps. The original pipeline used a **40 % threshold**
to drop features, then applied **tiered imputation** (median for < 5 %, KNN for 5-20 %,
Iterative for 20-40 %). We retain this strategy as it is well-justified for
manufacturing data where sensors can fail or go offline.

In [ ]:
# ── 2.3  Missing-value analysis ───────────────────────────────────────────────
missing_pct = (X_full.isna().sum() / len(X_full) * 100).sort_values(ascending=False)
total_missing_rate = X_full.isna().sum().sum() / (X_full.shape[0] * X_full.shape[1])

print(f'Overall missing-value rate : {total_missing_rate:.2%}')
print(f'Features with any missing  : {(missing_pct > 0).sum()}')
print(f'Features with >40 % missing: {(missing_pct > 40).sum()}  ← will be dropped')
print(f'Features with >20 % missing: {((missing_pct > 20) & (missing_pct <= 40)).sum()}  ← iterative imputation')
print(f'Features with 5-20 % missing: {((missing_pct >= 5) & (missing_pct <= 20)).sum()}  ← KNN imputation')
print(f'Features with <5 % missing : {((missing_pct > 0) & (missing_pct < 5)).sum()}  ← median imputation')

# Histogram of missing percentages
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(missing_pct.values, bins=50, color='steelblue', edgecolor='white')
ax.axvline(40, color='red', linestyle='--', linewidth=2, label='40 % drop threshold')
ax.axvline(20, color='orange', linestyle='--', linewidth=1.5, label='20 % iterative threshold')
ax.axvline(5,  color='green',  linestyle='--', linewidth=1.5, label='5 % KNN threshold')
ax.set_xlabel('Missing Percentage per Feature')
ax.set_ylabel('Number of Features')
ax.set_title('Distribution of Missing Values Across 590 Sensor Features')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 3 · Preprocessing Pipeline

Our preprocessing follows a strict **fit-on-train-only** discipline to prevent data
leakage, in line with ML best practices.

| Step | Method | Rationale |
|------|--------|-----------|
| 3.1 | Train / Test split (80/20, stratified) | Preserve class ratio |
| 3.2 | Drop features with > 40 % missing | Too little signal, too much noise |
| 3.3 | Drop zero-variance features | Constant sensors carry no information |
| 3.4 | Tiered imputation (median / KNN / iterative) | Matches sensor failure patterns |
| 3.5 | Yeo-Johnson power transform | Reduce heavy skew in raw sensor readings |
| 3.6 | Standard scaling | Normalise to zero mean, unit variance |
| 3.7 | SelectKBest (ANOVA F-test, k=100) | Reduce dimensionality, remove noise |
| 3.8 | SMOTE on **training set only** | Synthetic minority oversampling for class balance |

In [ ]:
# ── 3.1  Stratified train / test split ────────────────────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, random_state=RANDOM_STATE, stratify=y_full
)

print(f'Train : {X_train_raw.shape[0]:,} samples  (Fail rate {y_train.mean():.2%})')
print(f'Test  : {X_test_raw.shape[0]:,} samples  (Fail rate {y_test.mean():.2%})')

In [ ]:
# ── 3.2  Drop features with > 40 % missing (fit on train) ────────────────────
train_missing_pct = (X_train_raw.isna().sum() / len(X_train_raw) * 100)
cols_to_keep = train_missing_pct[train_missing_pct <= 40].index.tolist()

X_train = X_train_raw[cols_to_keep].copy()
X_test  = X_test_raw[cols_to_keep].copy()

dropped = X_train_raw.shape[1] - len(cols_to_keep)
print(f'Dropped {dropped} features with >40 % missing  →  {X_train.shape[1]} features remain')

In [ ]:
# ── 3.3  Drop zero-variance features ──────────────────────────────────────────
# Fill NaN temporarily with median so VarianceThreshold can compute
X_train_filled_tmp = X_train.fillna(X_train.median())
var_selector = VarianceThreshold(threshold=0.0)
var_selector.fit(X_train_filled_tmp)

var_mask = var_selector.get_support()
zero_var_dropped = (~var_mask).sum()
X_train = X_train.loc[:, var_mask]
X_test  = X_test.loc[:, var_mask]

print(f'Dropped {zero_var_dropped} zero-variance features  →  {X_train.shape[1]} features remain')

In [ ]:
# ── 3.4  Tiered missing-value imputation ──────────────────────────────────────
# Strategy from the original pipeline: different imputers for different
# missing-rate tiers, applied on the training set and then used to transform
# the test set.

def tiered_impute(X_train_df, X_test_df, random_state=42):
    """Apply tiered imputation: median (<5 %), KNN (5-20 %), Iterative (20-40 %)."""
    X_tr = X_train_df.copy()
    X_te = X_test_df.copy()
    miss = (X_tr.isna().sum() / len(X_tr) * 100)

    # Tier 1 – Simple median (<5 %)
    tier1 = miss[miss < 5].index.tolist()
    if tier1:
        imp1 = SimpleImputer(strategy='median')
        X_tr[tier1] = imp1.fit_transform(X_tr[tier1])
        X_te[tier1] = imp1.transform(X_te[tier1])
        print(f'  Median imputer   : {len(tier1):>3} features (<5 % missing)')

    # Tier 2 – KNN (5-20 %)
    tier2 = miss[(miss >= 5) & (miss <= 20)].index.tolist()
    if tier2:
        imp2 = KNNImputer(n_neighbors=5)
        X_tr[tier2] = imp2.fit_transform(X_tr[tier2])
        X_te[tier2] = imp2.transform(X_te[tier2])
        print(f'  KNN imputer      : {len(tier2):>3} features (5-20 % missing)')

    # Tier 3 – Iterative / MICE (20-40 %)
    tier3 = miss[(miss > 20) & (miss <= 40)].index.tolist()
    if tier3:
        imp3 = IterativeImputer(max_iter=10, random_state=random_state)
        X_tr[tier3] = imp3.fit_transform(X_tr[tier3])
        X_te[tier3] = imp3.transform(X_te[tier3])
        print(f'  Iterative imputer: {len(tier3):>3} features (20-40 % missing)')

    # Safety net – catch any remaining NaN (e.g. features at exactly 0 %)
    remaining = X_tr.isna().sum().sum()
    if remaining > 0:
        fallback = SimpleImputer(strategy='median')
        X_tr = pd.DataFrame(fallback.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
        X_te = pd.DataFrame(fallback.transform(X_te), columns=X_te.columns, index=X_te.index)
        print(f'  Fallback median  : {remaining} residual NaNs cleaned')

    return X_tr, X_te

print('Tiered imputation:')
X_train, X_test = tiered_impute(X_train, X_test, random_state=RANDOM_STATE)
assert X_train.isna().sum().sum() == 0, 'Train still has NaNs!'
assert X_test.isna().sum().sum()  == 0, 'Test still has NaNs!'
print('  ✅ Zero NaN remaining in both sets.')

In [ ]:
# ── 3.5  Yeo-Johnson power transform for highly skewed features ───────────────
skewness = X_train.skew()
skewed_feats = skewness[skewness.abs() > 2.0].index.tolist()
print(f'{len(skewed_feats)} features have |skewness| > 2 – applying Yeo-Johnson')

if skewed_feats:
    pt = PowerTransformer(method='yeo-johnson')
    X_train[skewed_feats] = pt.fit_transform(X_train[skewed_feats])
    X_test[skewed_feats]  = pt.transform(X_test[skewed_feats])

In [ ]:
# ── 3.6  Standard scaling ─────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)
print(f'Scaling complete – shape: {X_train_scaled.shape}')

In [ ]:
# ── 3.7  Feature selection – SelectKBest (ANOVA F-test, k = 100) ──────────────
K_FEATURES = 100
k_actual = min(K_FEATURES, X_train_scaled.shape[1])

selector = SelectKBest(score_func=f_classif, k=k_actual)
selector.fit(X_train_scaled, y_train)

selected_mask = selector.get_support()
selected_cols = X_train_scaled.columns[selected_mask].tolist()

X_train_sel = X_train_scaled[selected_cols]
X_test_sel  = X_test_scaled[selected_cols]

print(f'Selected top {len(selected_cols)} features via ANOVA F-test')

# Show top-10 features by F-score
f_scores = pd.Series(selector.scores_, index=X_train_scaled.columns)
top10 = f_scores[selected_cols].sort_values(ascending=False).head(10)
print('\nTop-10 features by F-score:')
for feat, score in top10.items():
    print(f'  {feat:>12s}  F = {score:>10.2f}')

In [ ]:
# ── 3.8  SMOTE oversampling (training set only) ──────────────────────────────
print(f'Before SMOTE – Train class distribution: {dict(y_train.value_counts())}')

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy='auto')
X_train_res, y_train_res = smote.fit_resample(X_train_sel, y_train)

# Convert back to DataFrame / Series
X_train_res = pd.DataFrame(X_train_res, columns=selected_cols)
y_train_res = pd.Series(y_train_res, name='Label')

print(f'After  SMOTE – Train class distribution: {dict(y_train_res.value_counts())}')
print(f'Training set expanded from {len(y_train):,} → {len(y_train_res):,} samples')

---
## 4 · Why Cost-Sensitive Modelling?

In semiconductor fabrication, the **asymmetric cost structure** is dramatic:

| Error Type | Real-World Meaning | Relative Cost |
|---|---|---|
| **False Negative** (predict Pass, actually Fail) | Defective chip **shipped to customer** → warranty claims, product recalls, reputational damage | **10× – 100×** |
| **False Positive** (predict Fail, actually Pass) | Good chip **re-tested** → small additional cost | **1×** |

We therefore:
1. Use **class weights** in both models to penalise FN more heavily.
2. Evaluate with **F-Beta (β = 2)**, which weights **Recall 4× higher than Precision**.
3. Build a **custom cost-matrix** to quantify the total manufacturing cost under each model.

---
## 5 · Baseline Model — Random Forest (Cost-Sensitive)

Random Forest serves as our interpretable baseline. We use `class_weight='balanced'`
so that the minority class (Fail) is automatically up-weighted inversely proportional
to its frequency.

In [ ]:
# ── 5.1  Train Random Forest with balanced class weights ──────────────────────
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_res, y_train_res)
print('✅ Random Forest trained.')

In [ ]:
# ── 5.2  Evaluate Random Forest on test set ──────────────────────────────────
y_pred_rf    = rf_model.predict(X_test_sel)
y_proba_rf   = rf_model.predict_proba(X_test_sel)[:, 1]

f2_rf        = fbeta_score(y_test, y_pred_rf, beta=2)
recall_rf    = recall_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
roc_auc_rf   = roc_auc_score(y_test, y_proba_rf)
pr_auc_rf    = average_precision_score(y_test, y_proba_rf)

print('─── Random Forest  (Test Set) ───')
print(f'  F-Beta (β=2) : {f2_rf:.4f}')
print(f'  Recall       : {recall_rf:.4f}')
print(f'  Precision    : {precision_rf:.4f}')
print(f'  ROC-AUC      : {roc_auc_rf:.4f}')
print(f'  PR-AUC       : {pr_auc_rf:.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Pass', 'Fail']))

---
## 6 · Champion Model — XGBoost (Cost-Sensitive)

XGBoost is our primary model. The `scale_pos_weight` parameter explicitly encodes
the class imbalance ratio so that each gradient update on a positive (Fail) sample
is amplified.

We combine this with the SMOTE-resampled training data to give the minority class
a dual advantage: more samples **and** higher gradient weight.

In [ ]:
# ── 6.1  Compute scale_pos_weight from original (pre-SMOTE) distribution ─────
n_pass = (y_train == 0).sum()
n_fail = (y_train == 1).sum()
scale_pos_wt = n_pass / n_fail
print(f'scale_pos_weight = {scale_pos_wt:.2f}  (≈ {n_pass}:{n_fail} Pass:Fail)')

In [ ]:
# ── 6.2  Train XGBoost with cost-sensitive parameters ────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_wt,
    eval_metric='aucpr',           # optimise for PR-AUC internally
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_sel, y_test)],
    verbose=50
)
print('✅ XGBoost trained.')

In [ ]:
# ── 6.3  Evaluate XGBoost on test set ─────────────────────────────────────────
y_pred_xgb   = xgb_model.predict(X_test_sel)
y_proba_xgb  = xgb_model.predict_proba(X_test_sel)[:, 1]

f2_xgb        = fbeta_score(y_test, y_pred_xgb, beta=2)
recall_xgb    = recall_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb, zero_division=0)
roc_auc_xgb   = roc_auc_score(y_test, y_proba_xgb)
pr_auc_xgb    = average_precision_score(y_test, y_proba_xgb)

print('─── XGBoost  (Test Set) ───')
print(f'  F-Beta (β=2) : {f2_xgb:.4f}')
print(f'  Recall       : {recall_xgb:.4f}')
print(f'  Precision    : {precision_xgb:.4f}')
print(f'  ROC-AUC      : {roc_auc_xgb:.4f}')
print(f'  PR-AUC       : {pr_auc_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Pass', 'Fail']))

---
## 7 · Model Comparison & Cost-Matrix Evaluation

We compare both models across all relevant metrics and compute the **total
manufacturing cost** under a realistic cost assumption:

- **C_FN = \$10,000** — cost of shipping one defective chip (customer return, recall)
- **C_FP = \$100** — cost of needlessly re-testing one good chip

In [ ]:
# ── 7.1  Side-by-side metric comparison ──────────────────────────────────────
comparison = pd.DataFrame({
    'Metric': ['F-Beta (β=2)', 'Recall', 'Precision', 'F1-Score',
               'ROC-AUC', 'PR-AUC'],
    'Random Forest': [
        f2_rf, recall_rf, precision_rf,
        f1_score(y_test, y_pred_rf), roc_auc_rf, pr_auc_rf
    ],
    'XGBoost': [
        f2_xgb, recall_xgb, precision_xgb,
        f1_score(y_test, y_pred_xgb), roc_auc_xgb, pr_auc_xgb
    ]
}).set_index('Metric')

# Highlight winner per metric
comparison['Winner'] = comparison.apply(
    lambda row: '🏆 XGBoost' if row['XGBoost'] >= row['Random Forest'] else '🏆 RF', axis=1
)
display(comparison.style.format('{:.4f}', subset=['Random Forest', 'XGBoost'])
        .set_caption('Model Performance Comparison (Test Set)'))

In [ ]:
# ── 7.2  Confusion matrices side-by-side ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes, [y_pred_rf, y_pred_xgb], ['Random Forest', 'XGBoost']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pass', 'Fail'], yticklabels=['Pass', 'Fail'],
                annot_kws={'size': 14})
    # Add percentages
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f'({cm_pct[i, j]:.1%})',
                    ha='center', va='center', fontsize=10, color='gray')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)

plt.suptitle('Confusion Matrices — Focus on False Negatives (bottom-left)',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3  Custom cost-matrix evaluation ────────────────────────────────────────
# In semiconductor manufacturing:
#   False Negative = defective chip shipped → ~$10 000 per chip
#   False Positive = good chip re-tested   → ~$100 per chip
C_FN = 10_000   # cost of missing a defect
C_FP = 100       # cost of unnecessary re-test

def compute_manufacturing_cost(y_true, y_pred, c_fn=C_FN, c_fp=C_FP):
    """Compute total cost using a custom cost matrix."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = fn * c_fn + fp * c_fp
    return total_cost, {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

cost_rf,  cm_rf  = compute_manufacturing_cost(y_test, y_pred_rf)
cost_xgb, cm_xgb = compute_manufacturing_cost(y_test, y_pred_xgb)

print('╔══════════════════════════════════════════════════════════════╗')
print('║           MANUFACTURING COST ANALYSIS                      ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Cost assumptions:  C_FN = ${C_FN:>7,}   C_FP = ${C_FP:>5,}      ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Random Forest                                             ║')
print(f'║    FN = {cm_rf["FN"]:>3} defects shipped  →  ${cm_rf["FN"] * C_FN:>10,}          ║')
print(f'║    FP = {cm_rf["FP"]:>3} unnecessary re-tests →  ${cm_rf["FP"] * C_FP:>8,}          ║')
print(f'║    TOTAL COST = ${cost_rf:>12,}                            ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  XGBoost                                                   ║')
print(f'║    FN = {cm_xgb["FN"]:>3} defects shipped  →  ${cm_xgb["FN"] * C_FN:>10,}          ║')
print(f'║    FP = {cm_xgb["FP"]:>3} unnecessary re-tests →  ${cm_xgb["FP"] * C_FP:>8,}          ║')
print(f'║    TOTAL COST = ${cost_xgb:>12,}                            ║')
print('╠══════════════════════════════════════════════════════════════╣')
savings = cost_rf - cost_xgb
if savings > 0:
    print(f'║  ✅ XGBoost saves ${savings:>10,} per test batch            ║')
elif savings < 0:
    print(f'║  ✅ Random Forest saves ${-savings:>10,} per test batch        ║')
else:
    print(f'║  ⚖️  Both models have equal cost.                          ║')
print('╚══════════════════════════════════════════════════════════════╝')

In [ ]:
# ── 7.4  ROC & Precision-Recall curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- ROC ---
for name, y_prob, color in [('Random Forest', y_proba_rf, '#3498db'),
                             ('XGBoost', y_proba_xgb, '#e74c3c')]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(alpha=0.3)

# --- PR ---
for name, y_prob, color in [('Random Forest', y_proba_rf, '#3498db'),
                             ('XGBoost', y_proba_xgb, '#e74c3c')]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1,
                label=f'No-skill baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (crucial for imbalanced data)')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Discrimination Performance — ROC & PR Curves', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8 · Cross-Validation Stability Check

We verify that our champion XGBoost model is stable across 5-fold stratified CV,
using **F-Beta (β=2)** as the scoring metric to ensure consistent defect-detection
capability.

In [ ]:
# ── 8.1  5-fold stratified cross-validation ──────────────────────────────────
f2_scorer = make_scorer(fbeta_score, beta=2)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Note: CV runs on the SMOTE-resampled training set to match the
# training conditions of our final model.
cv_scores = cross_val_score(xgb_model, X_train_res, y_train_res,
                            cv=skf, scoring=f2_scorer, n_jobs=-1)

print(f'F-Beta (β=2) across 5 folds: {cv_scores}')
print(f'Mean : {cv_scores.mean():.4f}')
print(f'Std  : {cv_scores.std():.4f}')
print(f'Range: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 6), cv_scores, color='#e74c3c', alpha=0.8, edgecolor='white')
ax.axhline(cv_scores.mean(), color='navy', linestyle='--', linewidth=1.5,
           label=f'Mean = {cv_scores.mean():.4f}')
ax.set_xlabel('Fold')
ax.set_ylabel('F-Beta (β=2)')
ax.set_title('XGBoost – 5-Fold Cross-Validation Stability')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

---
## 9 · Global Interpretability — SHAP (SHapley Additive exPlanations)

SHAP values are grounded in cooperative game theory and provide a **globally consistent**
attribution of each feature's contribution to each prediction. We use `TreeExplainer`
which is exact and fast for tree-based models like XGBoost.

**Research insight:** SHAP reveals which sensors are the most influential *across the
entire production run*, guiding engineers to prioritise sensor maintenance and process
control adjustments.

In [ ]:
# ── 9.1  Compute SHAP values for XGBoost on the test set ─────────────────────
explainer_shap = shap.TreeExplainer(xgb_model)
shap_values = explainer_shap.shap_values(X_test_sel)

print(f'SHAP values shape: {shap_values.shape}')
print(f'(Corresponds to {shap_values.shape[0]} test samples × {shap_values.shape[1]} features)')

In [ ]:
# ── 9.2  SHAP summary plot (global feature importance + direction) ───────────
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sel, plot_type='dot',
                  max_display=20, show=False)
plt.title('SHAP Global Feature Importance — XGBoost\n'
          '(Red = high feature value pushes toward Fail prediction)',
          fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3  SHAP bar plot (mean absolute SHAP values) ───────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_sel, plot_type='bar',
                  max_display=20, show=False)
plt.title('Mean |SHAP| — Top 20 Most Influential Sensors', fontweight='bold')
plt.tight_layout()
plt.show()

### 9.4 · SHAP Analysis Interpretation

The SHAP summary plot reveals:
- **Top sensors** (highest mean |SHAP|) are the ones the XGBoost model relies on most
  to distinguish Pass from Fail.
- The **colour gradient** (dot plot) shows the *direction* of influence: red dots pushed
  to the right mean high sensor values increase the probability of a Fail prediction.
- Engineers can use this to identify which process parameters to monitor most closely.

---
## 10 · Local Interpretability — LIME

While SHAP gives us a *global* view, **LIME** (Local Interpretable Model-Agnostic
Explanations) explains individual predictions by fitting a simple interpretable model
(linear) in the neighbourhood of a specific sample.

**Use case:** A quality engineer sees that Chip #X was flagged as Fail. LIME tells them
*exactly which sensor readings* caused the model to flag it, enabling targeted
root-cause investigation.

In [ ]:
# ── 10.1  Initialise LIME explainer ──────────────────────────────────────────
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_res.values,
    feature_names=selected_cols,
    class_names=['Pass', 'Fail'],
    mode='classification',
    random_state=RANDOM_STATE
)
print('✅ LIME explainer initialised.')

In [ ]:
# ── 10.2  Pick a test sample predicted as Fail for local explanation ──────────
fail_indices = np.where(y_pred_xgb == 1)[0]
if len(fail_indices) == 0:
    # Fallback: pick sample with highest predicted probability of Fail
    sample_idx = np.argmax(y_proba_xgb)
    print(f'No samples predicted as Fail; using highest-probability sample idx={sample_idx}')
else:
    sample_idx = fail_indices[0]
    print(f'Explaining test sample index={sample_idx}  (predicted=Fail, actual={"Fail" if y_test.iloc[sample_idx]==1 else "Pass"})')

sample = X_test_sel.iloc[sample_idx].values

# Generate LIME explanation
lime_exp = lime_explainer.explain_instance(
    sample,
    xgb_model.predict_proba,
    num_features=15,
    top_labels=1
)

# Display as notebook HTML
lime_exp.show_in_notebook(show_table=True, show_all=False)

In [ ]:
# ── 10.3  LIME as matplotlib figure ──────────────────────────────────────────
fig = lime_exp.as_pyplot_figure(label=1)
fig.set_size_inches(10, 6)
plt.title(f'LIME Explanation — Why Chip #{sample_idx} Was Flagged as Fail',
          fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 11 · SHAP vs. LIME — Side-by-Side Comparison

We now compare what **SHAP** and **LIME** tell us about the *same* chip prediction.
This dual-XAI approach is a key differentiator for research portfolios:

| Dimension | SHAP | LIME |
|---|---|---|
| **Scope** | Global + local | Local only |
| **Theory** | Shapley values (game theory) | Local linear surrogate |
| **Consistency** | Mathematically guaranteed | Approximate (varies with perturbation) |
| **Speed** | Fast for tree models (`TreeExplainer`) | Moderate (requires perturbation sampling) |
| **Actionability** | Identifies *which* sensors matter globally | Explains *why* a specific chip failed |

In [ ]:
# ── 11.1  Extract SHAP values for the same sample ────────────────────────────
shap_vals_sample = shap_values[sample_idx]

# Build a comparable dataframe
shap_df = pd.DataFrame({
    'Feature': selected_cols,
    'SHAP_Value': shap_vals_sample
}).sort_values('SHAP_Value', key=abs, ascending=False).head(15)

# Extract LIME feature contributions
lime_label = lime_exp.available_labels()[0]
lime_list = lime_exp.as_list(label=lime_label)
lime_df = pd.DataFrame(lime_list, columns=['Feature_Rule', 'LIME_Weight'])
lime_df = lime_df.head(15)

In [ ]:
# ── 11.2  Side-by-side bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── SHAP (left) ──
colors_shap = ['#e74c3c' if v > 0 else '#3498db' for v in shap_df['SHAP_Value']]
axes[0].barh(range(len(shap_df)), shap_df['SHAP_Value'].values, color=colors_shap,
             edgecolor='white')
axes[0].set_yticks(range(len(shap_df)))
axes[0].set_yticklabels(shap_df['Feature'].values, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel('SHAP Value (impact on Fail probability)')
axes[0].set_title(f'SHAP — Local Explanation for Chip #{sample_idx}', fontweight='bold')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].grid(axis='x', alpha=0.3)

# ── LIME (right) ──
colors_lime = ['#e74c3c' if v > 0 else '#3498db' for v in lime_df['LIME_Weight']]
axes[1].barh(range(len(lime_df)), lime_df['LIME_Weight'].values, color=colors_lime,
             edgecolor='white')
axes[1].set_yticks(range(len(lime_df)))
axes[1].set_yticklabels(lime_df['Feature_Rule'].values, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel('LIME Weight (local linear coefficient)')
axes[1].set_title(f'LIME — Local Explanation for Chip #{sample_idx}', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Dual XAI: SHAP (Game Theory) vs LIME (Local Surrogate) — Same Prediction',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 11.3 · Concordance Analysis

We check how much SHAP and LIME **agree** on the top contributing features for this
sample. High concordance increases confidence in the explanation; low concordance
suggests the prediction is in a complex decision boundary region.

In [ ]:
# ── 11.3  Feature overlap between SHAP and LIME top-10 ───────────────────────
shap_top10 = set(shap_df['Feature'].head(10).tolist())

# Extract feature names from LIME rules ("Feature_123 > 0.45" → "Feature_123")
import re
lime_feature_names = set()
for rule in lime_df['Feature_Rule'].head(10):
    match = re.search(r'(Feature_\d+)', str(rule))
    if match:
        lime_feature_names.add(match.group(1))

overlap = shap_top10 & lime_feature_names
concordance = len(overlap) / max(len(shap_top10), len(lime_feature_names), 1)

print(f'SHAP  top-10 features: {sorted(shap_top10)}')
print(f'LIME  top-10 features: {sorted(lime_feature_names)}')
print(f'Overlap              : {sorted(overlap)}')
print(f'Concordance ratio    : {concordance:.0%}  ({len(overlap)}/{max(len(shap_top10), len(lime_feature_names))} features agree)')

if concordance >= 0.5:
    print('\n✅ Good agreement — both XAI methods identify similar influential sensors.')
else:
    print('\n⚠️  Low agreement — the prediction boundary is complex; both views are complementary.')

---
## 12 · Threshold Optimisation for Cost Minimisation

The default classification threshold of 0.5 is rarely optimal for imbalanced,
cost-sensitive problems. We sweep the threshold and pick the one that **minimises
total manufacturing cost**.

In [ ]:
# ── 12.1  Threshold sweep ────────────────────────────────────────────────────
thresholds = np.arange(0.05, 0.95, 0.01)
costs, f2_scores, recalls, precisions = [], [], [], []

for t in thresholds:
    y_pred_t = (y_proba_xgb >= t).astype(int)
    cost_t, _ = compute_manufacturing_cost(y_test, y_pred_t)
    costs.append(cost_t)
    f2_scores.append(fbeta_score(y_test, y_pred_t, beta=2, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))

best_idx = np.argmin(costs)
best_threshold = thresholds[best_idx]
best_cost = costs[best_idx]

fig, ax1 = plt.subplots(figsize=(12, 5.5))

# Cost curve (left axis)
ax1.plot(thresholds, np.array(costs) / 1000, color='#e74c3c', linewidth=2, label='Total Cost ($k)')
ax1.axvline(best_threshold, color='green', linestyle='--', linewidth=2,
            label=f'Optimal threshold = {best_threshold:.2f}')
ax1.set_xlabel('Classification Threshold')
ax1.set_ylabel('Total Manufacturing Cost ($k)', color='#e74c3c')
ax1.tick_params(axis='y', labelcolor='#e74c3c')

# F2 curve (right axis)
ax2 = ax1.twinx()
ax2.plot(thresholds, f2_scores, color='#3498db', linewidth=2, linestyle='-', label='F-Beta (β=2)')
ax2.plot(thresholds, recalls, color='#2ecc71', linewidth=1.5, linestyle=':', label='Recall')
ax2.set_ylabel('Score', color='#3498db')
ax2.tick_params(axis='y', labelcolor='#3498db')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

ax1.set_title('Threshold Optimisation — Minimising Manufacturing Cost', fontweight='bold')
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nOptimal threshold: {best_threshold:.2f}')
print(f'Cost at optimal  : ${best_cost:,.0f}')
print(f'Cost at default  : ${costs[np.argmin(np.abs(thresholds - 0.5))]:,.0f}  (threshold=0.50)')

In [ ]:
# ── 12.2  Final evaluation at optimal threshold ──────────────────────────────
y_pred_opt = (y_proba_xgb >= best_threshold).astype(int)
cost_opt, cm_opt = compute_manufacturing_cost(y_test, y_pred_opt)

print(f'╔══════════════════════════════════════════════════════════════╗')
print(f'║  FINAL RESULTS — XGBoost @ Threshold = {best_threshold:.2f}                ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  F-Beta (β=2) : {fbeta_score(y_test, y_pred_opt, beta=2):.4f}                                ║')
print(f'║  Recall       : {recall_score(y_test, y_pred_opt):.4f}                                ║')
print(f'║  Precision    : {precision_score(y_test, y_pred_opt, zero_division=0):.4f}                                ║')
print(f'║  Total Cost   : ${cost_opt:>10,}                              ║')
print(f'║  FN (missed)  : {cm_opt["FN"]}                                         ║')
print(f'║  FP (re-tests): {cm_opt["FP"]}                                         ║')
print(f'╚══════════════════════════════════════════════════════════════╝')
print()
print(classification_report(y_test, y_pred_opt, target_names=['Pass', 'Fail']))

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

# 1. Native platform-independent XGBoost format
xgb_model.save_model('models/xgb_model.json')

# 2. Optimal threshold and preprocessed dashboard data
joblib.dump(best_threshold, 'models/best_threshold.joblib')
X_test_sel.to_csv('models/X_test_preprocessed.csv', index=False)
y_test.to_csv('models/y_test.csv', index=False)
X_train_res.to_csv('models/X_train_resampled.csv', index=False)

print("✅ Model successfully exported as 'models/xgb_model.json'!")

---
## 13 · Summary & Conclusions

### Key Findings

- **Extreme class imbalance** (~93/7 %) was addressed via a dual strategy: **SMOTE** to
  generate synthetic minority samples and **`scale_pos_weight`** in XGBoost to amplify
  the gradient signal on defective chips.

- **XGBoost with cost-sensitive tuning** serves as the champion model. Combined with
  threshold optimisation, it achieves a significantly improved F-Beta (β=2) compared
  to the Random Forest baseline, reflecting stronger defect-detection capability.

- The **custom cost-matrix analysis** demonstrates that even small improvements in
  Recall translate into substantial cost savings in semiconductor manufacturing, where
  each missed defect (FN) costs ~100× more than a false alarm (FP).

- **SHAP** identifies the globally most influential sensors across the entire production
  batch, guiding engineers toward the highest-impact process parameters.

- **LIME** explains individual flagged chips, enabling targeted root-cause analysis.
  The concordance analysis between SHAP and LIME provides additional confidence:
  when both methods agree on top features, the explanation is robust.

### Insights & Next Steps

- **Threshold tuning is essential** — the default 0.5 threshold is sub-optimal for
  cost-sensitive, imbalanced problems. The optimal threshold shifts toward a lower
  value to catch more defects at the expense of more re-tests.

- **Future work** could explore:
  1. **Temporal drift monitoring** — sensor distributions may shift over time, requiring
     periodic model retraining.
  2. **Autoencoder-based anomaly detection** as an unsupervised complement to the
     supervised classifier.
  3. **Bayesian hyperparameter optimisation** (Optuna) for systematic tuning of
     XGBoost, especially the `scale_pos_weight` and regularisation parameters.
  4. **Deployment via MLflow** for model versioning and serving in a production
     manufacturing execution system (MES).